# HCMAN reviewer training: reference set, practice, qualification

Runs the practice and qualification parts of the **HCMAN Label Review Training Program** inside the
Mayo Clinic Platform workspace. Every set is real de-identified data, so nothing here leaves the workspace.

| Step | Who | Section |
| --- | --- | --- |
| 1. Draw the reference set and practice pool | Run owner | A |
| 2. Write the reference answers | Clinical lead | B |
| 3. Practice with feedback | Each trainee | C |
| 4. Qualification (attempt 1, retake = attempt 2) | Each trainee | D |
| 5. Score and record the result | Run owner | E |
| 6. Monthly refresher | Each qualified reviewer | F |

Run Steps 1–3 of the RUNBOOK first (study tables built) and Step 2b's automatic echo pass (`annotate.echo auto`),
because the sets are drawn from the automatic labels. Do this **before** drawing the live audit samples, so the
training women and clips can be kept out of them.

In [ ]:
import sys; sys.path.insert(0, "..")          # the ml/ folder
import pandas as pd
from mcp_adapter.config import StudyConfig
from annotate.store import AnnotationStore
from annotate.training import (TrainingScreen, draw_training_sets, exclude_training, save_training_sets,
                               load_training_sets, qualification_report, pair_kappa, refresher_items,
                               marks_from_config)

cfg = StudyConfig.load("../config/study.yaml")
WB = "/workbench"                              # your workbench folder
LEAD = cfg.training.get("lead")                # clinical lead's user name (config: training.lead)
SETS = f"{WB}/annot/training_sets.csv"
STORE = AnnotationStore(f"{WB}/annot/training.csv")    # training answers only, never the live audit store
CTX = f"{WB}/annot/training_context.pkl"
CACHE = f"{WB}/annot/cache"                    # echo cache from annotate.echo auto
assert LEAD, "set training.lead in config/study.yaml"

## A. Run owner: draw the sets (once)

Reference set: 40 records (conditions and outcomes) and 40 echo clips, split into halves A and B
(qualification attempt 1 and the retake). Practice pool: 60 + 60. Sizes come from `training:` in the config.

In [ ]:
from mcp_adapter.backend import SparkBackend
from annotate.phenotype import fetch_context, save_context

be = SparkBackend(spark)                       # the MCP notebook provides `spark`
labels = pd.read_pickle(f"{WB}/hcman_study/labels.pkl")
ev = pd.read_pickle(f"{WB}/hcman_study/code_events.pkl")
cohort = pd.read_pickle(f"{WB}/hcman_study/cohort.pkl")
echo_store = AnnotationStore(f"{WB}/annot/echo.csv")

t = cfg.training
sets = draw_training_sets(labels, ev, echo_store,
                          n_reference=t["reference_size"]["records"], n_practice=t["practice_size"]["records"],
                          n_echo_reference=t["reference_size"]["echo"], n_echo_practice=t["practice_size"]["echo"],
                          seed=cfg.seed)
save_training_sets(sets, SETS)
women = sets.loc[sets.task != "echo", "item_id"].astype(int).unique()
save_context(fetch_context(be, cfg, cohort, women), CTX)
sets.groupby(["set", "part", "task"]).size()

**Keep the training items out of the live audit.** In RUNBOOK Steps 2b and 4b, filter the live samples:

```python
from annotate.training import exclude_training, load_training_sets
training = load_training_sets(f"{WB}/annot/training_sets.csv")
s = exclude_training(s, training)                                   # Step 4b chart-review sample
items = exclude_training(echo_audit_items(echo_store), training)    # Step 2b echo audit items
```

## B. Clinical lead: write the reference answers

Blinded: no automatic label is shown. Answer every record and clip, and write a note on each one;
trainees see the note as feedback during practice.

In [ ]:
sets = load_training_sets(SETS)
TrainingScreen(sets, STORE, LEAD, "reference", phenotype_context=CTX, echo_cache=CACHE).show()

## C. Trainee: practice with feedback

After each answer the screen shows the reference answer and the lead's note. Work through the whole pool
before qualifying; it may be repeated in discussion with the lead.

In [ ]:
ME = "<your user name>"
TrainingScreen(sets, STORE, ME, "practice", phenotype_context=CTX, echo_cache=CACHE, lead=LEAD).show()

## D. Trainee: qualification

Attempt 1 uses reference half A. No feedback is shown. A trainee who misses a mark reviews their errors with
the lead, then takes attempt 2 (half B) once.

In [ ]:
TrainingScreen(sets, STORE, ME, "qualification", attempt=1, phenotype_context=CTX, echo_cache=CACHE, lead=LEAD).show()

## E. Run owner: score

Pass marks come from `training.pass_marks` in the config. The report holds reviewer names and scores only.
Two cardiologists must also agree with each other (Cohen's kappa at least 0.80).

In [ ]:
marks = marks_from_config(cfg)
trainees = {"<ds user 1>": "data_scientist", "<ds user 2>": "data_scientist",
            "<cardiologist user>": "cardiologist"}
rep = qualification_report(STORE, sets, trainees, attempt=1, marks=marks, lead=LEAD)
rep.to_csv(f"{WB}/annot/qualification_attempt1.csv", index=False)
rep

In [ ]:
# only when two cardiologists adjudicate
pair = pair_kappa(STORE, sets, "<cardiologist 1>", "<cardiologist 2>", attempt=1)
pair, pair["kappa"] >= marks["cardiologist_pair_kappa"]

Only reviewers with `passed = True` are added to `audit.data_scientists` / `audit.cardiologists` in the config.

## F. Monthly refresher

Each month of active audit, every reviewer labels 10 reference items with no feedback. The run owner compares
them with the reference answers; a reviewer who falls below their pass rate repeats practice.

In [ ]:
MONTH = 1
items = refresher_items(sets, MONTH)
TrainingScreen(sets, STORE, ME, "refresher", items=items, month=MONTH,
               phenotype_context=CTX, echo_cache=CACHE, lead=LEAD).show()

In [ ]:
from annotate.training import refresher_report
refresher_report(STORE, sets, MONTH, list(trainees), lead=LEAD)
